In [10]:
import pandas as pd

df = pd.read_csv("daily_performance_report.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   date                        37 non-null     object 
 1   controller_id               37 non-null     object 
 2   market_volume_usdt          37 non-null     float64
 3   market_price_min            37 non-null     float64
 4   market_price_max            37 non-null     float64
 5   market_price_var_pct        37 non-null     float64
 6   market_trades_count         37 non-null     int64  
 7   bot_volume_usdt             37 non-null     float64
 8   bot_volume_base             37 non-null     float64
 9   bot_realized_pnl            37 non-null     float64
 10  bot_unrealized_pnl          37 non-null     float64
 11  bot_trades_count            37 non-null     int64  
 12  bot_orphan_trades_count     37 non-null     int64  
 13  bot_market_share_pct        37 non-nu

In [11]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Prepare data
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

# Aggregate by date for daily metrics (in case multiple controllers per day)
daily_agg = df.groupby('date').agg({
    'bot_realized_pnl': 'first',
    'bot_unrealized_pnl': 'first',
    'bot_volume_usdt': 'first',
    'bot_trades_count': 'first',
    'bot_market_share_pct': 'first',
    'market_volume_usdt': 'first',
    'controller_realized_pnl': 'sum',
    'controller_volume_usdt': 'sum',
}).reset_index()

# Calculate cumulative PnL
daily_agg['cumulative_realized_pnl'] = daily_agg['bot_realized_pnl'].cumsum()

# Create executive dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Cumulative Realized P&L',
        'Daily Trading Volume',
        'Market Share %',
        'Daily P&L (Realized vs Unrealized)',
        'Trading Activity',
        'Daily Returns'
    ),
    specs=[
        [{"secondary_y": False}, {"secondary_y": False}],
        [{"secondary_y": False}, {"secondary_y": False}],
        [{"secondary_y": False}, {"secondary_y": False}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12
)

# 1. Cumulative Realized P&L (most important metric)
fig.add_trace(
    go.Scatter(
        x=daily_agg['date'],
        y=daily_agg['cumulative_realized_pnl'],
        mode='lines+markers',
        name='Cumulative P&L',
        line=dict(color='#2E86AB', width=3),
        fill='tozeroy',
        fillcolor='rgba(46, 134, 171, 0.2)'
    ),
    row=1, col=1
)

# 2. Daily Trading Volume
fig.add_trace(
    go.Bar(
        x=daily_agg['date'],
        y=daily_agg['bot_volume_usdt'],
        name='Bot Volume',
        marker_color='#06A77D'
    ),
    row=1, col=2
)

# 3. Market Share %
fig.add_trace(
    go.Scatter(
        x=daily_agg['date'],
        y=daily_agg['bot_market_share_pct'],
        mode='lines+markers',
        name='Market Share',
        line=dict(color='#F77F00', width=2),
        marker=dict(size=8)
    ),
    row=2, col=1
)

# 4. Daily P&L (Realized vs Unrealized)
fig.add_trace(
    go.Bar(
        x=daily_agg['date'],
        y=daily_agg['bot_realized_pnl'],
        name='Realized P&L',
        marker_color='#2E86AB'
    ),
    row=2, col=2
)
fig.add_trace(
    go.Bar(
        x=daily_agg['date'],
        y=daily_agg['bot_unrealized_pnl'],
        name='Unrealized P&L',
        marker_color='#D62828',
        opacity=0.6
    ),
    row=2, col=2
)

# 5. Trading Activity
fig.add_trace(
    go.Bar(
        x=daily_agg['date'],
        y=daily_agg['bot_trades_count'],
        name='Trades Count',
        marker_color='#A23B72'
    ),
    row=3, col=1
)

# 6. Daily Returns (P&L as % of volume)
daily_agg['daily_return_pct'] = (daily_agg['bot_realized_pnl'] / daily_agg['bot_volume_usdt'] * 100).replace([np.inf, -np.inf], 0).fillna(0)
colors = ['#06A77D' if x >= 0 else '#D62828' for x in daily_agg['daily_return_pct']]

fig.add_trace(
    go.Bar(
        x=daily_agg['date'],
        y=daily_agg['daily_return_pct'],
        name='Daily Return %',
        marker_color=colors
    ),
    row=3, col=2
)

# Update axes labels
fig.update_xaxes(title_text="Date", row=3, col=1)
fig.update_xaxes(title_text="Date", row=3, col=2)

fig.update_yaxes(title_text="USDT", row=1, col=1)
fig.update_yaxes(title_text="USDT", row=1, col=2)
fig.update_yaxes(title_text="%", row=2, col=1)
fig.update_yaxes(title_text="USDT", row=2, col=2)
fig.update_yaxes(title_text="Count", row=3, col=1)
fig.update_yaxes(title_text="%", row=3, col=2)

# Update layout
fig.update_layout(
    title={
        'text': 'Trading Performance Executive Dashboard',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 24, 'color': '#1F1F1F'}
    },
    height=1200,
    showlegend=True,
    template='plotly_white',
    font=dict(size=11),
    hovermode='x unified'
)

fig.show()

In [12]:
# Risk Metrics Dashboard
daily_agg = df.groupby('date').agg({
    'bot_realized_pnl': 'first',
    'bot_volume_usdt': 'first',
}).reset_index()

daily_agg['date'] = pd.to_datetime(daily_agg['date'])
daily_agg = daily_agg.sort_values('date')

# Calculate risk metrics
daily_returns = daily_agg['bot_realized_pnl'].pct_change().dropna()
sharpe_ratio = (daily_returns.mean() / daily_returns.std() * np.sqrt(252)) if daily_returns.std() > 0 else 0

# Max Drawdown
cumulative_pnl = daily_agg['bot_realized_pnl'].cumsum()
running_max = cumulative_pnl.cummax()
drawdown = cumulative_pnl - running_max
max_drawdown = drawdown.min()
max_drawdown_pct = (max_drawdown / running_max.max() * 100) if running_max.max() > 0 else 0

# Volatility
volatility = daily_agg['bot_realized_pnl'].std()
volatility_pct = (volatility / daily_agg['bot_realized_pnl'].mean() * 100) if daily_agg['bot_realized_pnl'].mean() != 0 else 0

# Create risk dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Drawdown Analysis',
        'P&L Distribution',
        'Volume Consistency',
        'Rolling 7-Day Performance'
    ),
    specs=[[{"secondary_y": False}, {"type": "histogram"}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# 1. Drawdown
fig.add_trace(
    go.Scatter(
        x=daily_agg['date'],
        y=drawdown,
        fill='tozeroy',
        fillcolor='rgba(214, 40, 40, 0.3)',
        line=dict(color='#D62828', width=2),
        name='Drawdown'
    ),
    row=1, col=1
)

# 2. P&L Distribution
fig.add_trace(
    go.Histogram(
        x=daily_agg['bot_realized_pnl'],
        nbinsx=20,
        marker_color='#2E86AB',
        name='P&L Distribution'
    ),
    row=1, col=2
)

# 3. Volume Consistency
fig.add_trace(
    go.Scatter(
        x=daily_agg['date'],
        y=daily_agg['bot_volume_usdt'],
        mode='lines+markers',
        line=dict(color='#06A77D', width=2),
        name='Daily Volume'
    ),
    row=2, col=1
)

# Add moving average
ma_7 = daily_agg['bot_volume_usdt'].rolling(window=7).mean()
fig.add_trace(
    go.Scatter(
        x=daily_agg['date'],
        y=ma_7,
        mode='lines',
        line=dict(color='#F77F00', width=2, dash='dash'),
        name='7-Day MA'
    ),
    row=2, col=1
)

# 4. Rolling 7-day P&L
rolling_pnl = daily_agg['bot_realized_pnl'].rolling(window=7).sum()
fig.add_trace(
    go.Scatter(
        x=daily_agg['date'],
        y=rolling_pnl,
        mode='lines',
        line=dict(color='#2E86AB', width=2),
        fill='tozeroy',
        fillcolor='rgba(46, 134, 171, 0.2)',
        name='7-Day Rolling P&L'
    ),
    row=2, col=2
)

# Update axes
fig.update_yaxes(title_text="USDT", row=1, col=1)
fig.update_yaxes(title_text="Frequency", row=1, col=2)
fig.update_yaxes(title_text="USDT", row=2, col=1)
fig.update_yaxes(title_text="USDT", row=2, col=2)

fig.update_layout(
    title={
        'text': 'Risk & Performance Metrics',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    height=800,
    showlegend=True,
    template='plotly_white'
)

fig.show()

# Print risk metrics
print("\n" + "="*60)
print("RISK METRICS")
print("="*60)
print(f"Max Drawdown: ${max_drawdown:.2f} USDT ({max_drawdown_pct:.2f}%)")
print(f"Daily Volatility: ${volatility:.2f} USDT ({volatility_pct:.2f}%)")
print(f"Sharpe Ratio (Annualized): {sharpe_ratio:.3f}")
print(f"Best Day: ${daily_agg['bot_realized_pnl'].max():.2f} USDT")
print(f"Worst Day: ${daily_agg['bot_realized_pnl'].min():.2f} USDT")
print("="*60)


RISK METRICS
Max Drawdown: $-167234.19 USDT (0.00%)
Daily Volatility: $2550.81 USDT (-56.26%)
Sharpe Ratio (Annualized): 7.575
Best Day: $-507.05 USDT
Worst Day: $-9152.64 USDT


In [13]:
# Controller Performance Comparison
controller_summary = df.groupby('controller_id').agg({
    'controller_realized_pnl': 'sum',
    'controller_volume_usdt': 'sum',
    'controller_trades_count': 'sum',
}).reset_index()

controller_summary['roi_pct'] = (
    controller_summary['controller_realized_pnl'] / 
    controller_summary['controller_volume_usdt'] * 100
).replace([np.inf, -np.inf], 0).fillna(0)

controller_summary = controller_summary.sort_values('controller_realized_pnl', ascending=False)

# Create comparison chart
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'P&L by Controller',
        'Volume by Controller',
        'ROI % by Controller',
        'Trade Efficiency (P&L per Trade)'
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

# P&L by Controller
colors_pnl = ['#06A77D' if x >= 0 else '#D62828' for x in controller_summary['controller_realized_pnl']]
fig.add_trace(
    go.Bar(
        x=controller_summary['controller_id'],
        y=controller_summary['controller_realized_pnl'],
        marker_color=colors_pnl,
        name='P&L',
        text=controller_summary['controller_realized_pnl'].round(2),
        textposition='outside'
    ),
    row=1, col=1
)

# Volume by Controller
fig.add_trace(
    go.Bar(
        x=controller_summary['controller_id'],
        y=controller_summary['controller_volume_usdt'],
        marker_color='#2E86AB',
        name='Volume',
        text=controller_summary['controller_volume_usdt'].round(0),
        textposition='outside'
    ),
    row=1, col=2
)

# ROI % by Controller
colors_roi = ['#06A77D' if x >= 0 else '#D62828' for x in controller_summary['roi_pct']]
fig.add_trace(
    go.Bar(
        x=controller_summary['controller_id'],
        y=controller_summary['roi_pct'],
        marker_color=colors_roi,
        name='ROI %',
        text=controller_summary['roi_pct'].round(4),
        textposition='outside'
    ),
    row=2, col=1
)

# P&L per Trade
controller_summary['pnl_per_trade'] = (
    controller_summary['controller_realized_pnl'] / 
    controller_summary['controller_trades_count']
).replace([np.inf, -np.inf], 0).fillna(0)

colors_eff = ['#06A77D' if x >= 0 else '#D62828' for x in controller_summary['pnl_per_trade']]
fig.add_trace(
    go.Bar(
        x=controller_summary['controller_id'],
        y=controller_summary['pnl_per_trade'],
        marker_color=colors_eff,
        name='P&L per Trade',
        text=controller_summary['pnl_per_trade'].round(3),
        textposition='outside'
    ),
    row=2, col=2
)

# Update layout
fig.update_yaxes(title_text="USDT", row=1, col=1)
fig.update_yaxes(title_text="USDT", row=1, col=2)
fig.update_yaxes(title_text="%", row=2, col=1)
fig.update_yaxes(title_text="USDT", row=2, col=2)

fig.update_layout(
    title={
        'text': 'Controller Performance Comparison',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    height=800,
    showlegend=False,
    template='plotly_white'
)

fig.show()

In [14]:
# Calculate key summary metrics
total_realized_pnl = df['bot_realized_pnl'].iloc[-1]
total_volume = df['bot_volume_usdt'].sum()
total_trades = df['bot_trades_count'].sum()
avg_market_share = df['bot_market_share_pct'].mean()
num_days = len(df['date'].unique())
num_controllers = df['controller_id'].nunique()

# Calculate ROI
avg_daily_volume = total_volume / num_days
roi_pct = (total_realized_pnl / total_volume * 100) if total_volume > 0 else 0

# Win rate (days with positive PnL)
daily_pnl = df.groupby('date')['bot_realized_pnl'].first()
win_days = (daily_pnl > 0).sum()
win_rate = (win_days / len(daily_pnl) * 100) if len(daily_pnl) > 0 else 0

print("="*60)
print("EXECUTIVE SUMMARY")
print("="*60)
print(f"\n📊 Period: {num_days} days ({df['date'].min()} to {df['date'].max()})")
print(f"🤖 Active Controllers: {num_controllers}")
print(f"\n💰 FINANCIAL PERFORMANCE")
print(f"   Total Realized P&L: ${total_realized_pnl:,.2f} USDT")
print(f"   Total Volume Traded: ${total_volume:,.2f} USDT")
print(f"   Return on Volume: {roi_pct:.4f}%")
print(f"   Avg Daily Volume: ${avg_daily_volume:,.2f} USDT")
print(f"\n📈 TRADING ACTIVITY")
print(f"   Total Trades: {total_trades:,}")
print(f"   Avg Trades/Day: {total_trades/num_days:.1f}")
print(f"   Win Rate: {win_rate:.1f}% ({win_days}/{len(daily_pnl)} days)")
print(f"\n🎯 MARKET POSITION")
print(f"   Avg Market Share: {avg_market_share:.4f}%")
print("="*60)

EXECUTIVE SUMMARY

📊 Period: 37 days (2025-12-23 00:00:00 to 2026-01-28 00:00:00)
🤖 Active Controllers: 1

💰 FINANCIAL PERFORMANCE
   Total Realized P&L: $-9,151.05 USDT
   Total Volume Traded: $63,899,504.02 USDT
   Return on Volume: -0.0143%
   Avg Daily Volume: $1,727,013.62 USDT

📈 TRADING ACTIVITY
   Total Trades: 24,977
   Avg Trades/Day: 675.1
   Win Rate: 0.0% (0/37 days)

🎯 MARKET POSITION
   Avg Market Share: 6.3546%


# Executive Performance Dashboard

**Key Metrics Overview**